In [108]:
#!pip3 install torch tqdm jieba

import torch
from importlib import reload

import mt_transformer_zh_en as mt
reload(mt)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [109]:
import os
os.makedirs("data", exist_ok=True)

import pandas as pd

train_df = pd.read_csv("data/train.tsv", sep="\t")
valid_df = pd.read_csv("data/valid.tsv", sep="\t")

print(f"Train examples: {len(train_df):,}")
print(f"Valid examples: {len(valid_df):,}")
print(f"\nSample training examples:")
print(train_df.head(3))

Train examples: 46,466
Valid examples: 5,162

Sample training examples:
                                                  zh  \
0  当然 以下是返回您请求的格式的更新示例 在此示例中 查询的结果存储在字典中 其中键是 值是包...   
1  当然 这里有一份搬家公司的发票单样本 您的商标 您的公司名称 您的公司地址 您的公司电话号码...   
2                                            把钥匙扔出窗户   

                                                  en  
0  sure here's an updated example that returns th...  
1  sure here's an example invoice form for a movi...  
2                               throw key out window  


In [ ]:
train_limit = 43000   #2/3 size
valid_limit = 4800

train_src, train_tgt = mt.load_parallel_tsv("data/train.tsv", "zh", "en", limit=train_limit)
valid_src, valid_tgt = mt.load_parallel_tsv("data/valid.tsv", "zh", "en", limit=valid_limit)

print(f"Loaded {len(train_src):,} training pairs (subset)")
print(f"Loaded {len(valid_src):,} validation pairs (subset)")

src_tokens = [mt.tokenize_zh(s) for s in train_src]
tgt_tokens = [mt.tokenize_en(t) for t in train_tgt]

src_vocab = mt.Vocab(min_freq=2)
tgt_vocab = mt.Vocab(min_freq=2)
src_vocab.build(src_tokens)
tgt_vocab.build(tgt_tokens)

print(f"Source vocab size: {len(src_vocab.itos):,}")
print(f"Target vocab size: {len(tgt_vocab.itos):,}")

max_len = 64
batch_size = 32

train_ds = mt.TranslationDataset(
    train_src,
    train_tgt,
    mt.tokenize_zh,
    mt.tokenize_en,
    src_vocab,
    tgt_vocab,
    max_len=max_len,
)
valid_ds = mt.TranslationDataset(
    valid_src,
    valid_tgt,
    mt.tokenize_zh,
    mt.tokenize_en,
    src_vocab,
    tgt_vocab,
    max_len=max_len,
)

from torch.utils.data import DataLoader

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda b: mt.collate_fn(b, src_vocab.pad_idx, tgt_vocab.pad_idx),
)
valid_dl = DataLoader(
    valid_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda b: mt.collate_fn(b, src_vocab.pad_idx, tgt_vocab.pad_idx),
)

len(src_vocab.itos), len(tgt_vocab.itos)

Loaded 43,000 training pairs (subset)
Loaded 4,800 validation pairs (subset)
Source vocab size: 19,098
Target vocab size: 25,515


(19098, 25515)

In [111]:
emb_size = 256
ff_dim = 512
layers = 3
heads = 8

model = mt.Seq2SeqTransformer(
    num_encoder_layers=layers,
    num_decoder_layers=layers,
    emb_size=emb_size,
    nhead=heads,
    src_vocab_size=len(src_vocab.itos),
    tgt_vocab_size=len(tgt_vocab.itos),
    dim_feedforward=ff_dim,
    dropout=0.1,
    max_len=max_len + 10,
).to(device)

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    factor=0.7,
    patience=1,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 15,394,560


/Users/haley/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [112]:
import math

num_epochs = 10
best_val_loss = float("inf")
no_improve_epochs = 0
patience = 3

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")
    train_loss = mt.train_epoch(
        model,
        train_dl,
        optimizer,
        loss_fn,
        src_vocab.pad_idx,
        tgt_vocab.pad_idx,
        device,
    )
    val_loss = mt.evaluate_epoch(
        model,
        valid_dl,
        loss_fn,
        src_vocab.pad_idx,
        tgt_vocab.pad_idx,
        device,
    )
    print(f"  Train loss: {train_loss:.4f}, ppl: {math.exp(train_loss):.2f}")
    print(f"  Valid loss: {val_loss:.4f}, ppl: {math.exp(val_loss):.2f}")

    scheduler.step(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        no_improve_epochs = 0
        print("  (new best; saving checkpoint)")
        torch.save(
            {
                "model_state": model.state_dict(),
                "src_vocab": src_vocab,
                "tgt_vocab": tgt_vocab,
            },
            "transformer_zh_en_notebook.pt",
        )
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print("Early stopping triggered.")
            break


Epoch 1/10


Train:   0%|          | 0/1344 [00:00<?, ?it/s]/Users/haley/Library/Python/3.9/lib/python/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  Train loss: 6.1906, ppl: 488.12
  Valid loss: 5.4328, ppl: 228.79
  (new best; saving checkpoint)

Epoch 2/10


  Train loss: 5.2174, ppl: 184.45
  Valid loss: 4.8003, ppl: 121.55
  (new best; saving checkpoint)

Epoch 3/10


  Train loss: 4.6611, ppl: 105.75
  Valid loss: 4.3287, ppl: 75.84
  (new best; saving checkpoint)

Epoch 4/10


...

  Train loss: 3.4855, ppl: 32.64
  Valid loss: 3.4117, ppl: 30.32
  (new best; saving checkpoint)

Epoch 7/10


  Train loss: 3.2375, ppl: 25.47
  Valid loss: 3.2636, ppl: 26.14
  (new best; saving checkpoint)

Epoch 8/10


  Train loss: 3.0385, ppl: 20.87
  Valid loss: 3.1465, ppl: 23.25
  (new best; saving checkpoint)

Epoch 9/10


  Train loss: 2.8738, ppl: 17.70
  Valid loss: 3.0714, ppl: 21.57
  (new best; saving checkpoint)

Epoch 10/10


  Train loss: 2.7353, ppl: 15.41
  Valid loss: 3.0134, ppl: 20.36
  (new best; saving checkpoint)


In [114]:
model.eval()

test_sentences = [
    "今天天气很好。",
    "我喜欢机器学习。",
    "两个人在公园里散步。",
]

for test_sentence in test_sentences:
    tok = mt.tokenize_zh(test_sentence)
    ids = src_vocab.numericalize(tok, max_len=max_len - 2)
    src_tensor = torch.tensor([ids], dtype=torch.long)

    with torch.no_grad():
        pred_ids = mt.greedy_decode(
            model,
            src_tensor,
            src_pad_idx=src_vocab.pad_idx,
            bos_idx=tgt_vocab.bos_idx,
            eos_idx=tgt_vocab.eos_idx,
            max_len=max_len,
            device=device,
        )

    pred_tokens = mt.ids_to_tokens(pred_ids[0].tolist(), tgt_vocab)
    print(f"ZH: {test_sentence}")
    print(f"EN: {' '.join(pred_tokens)}\n")

ZH: 今天天气很好。
EN: <unk> is good

ZH: 我喜欢机器学习。
EN: i like machine learning

ZH: 两个人在公园里散步。
EN: two people at a park

